In [1]:
import asyncio
import httpx
import logging
import json
import sys
import os
from pathlib import Path
import time

In [4]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger("CreditCardTest")


In [43]:
request_data = {
        "scenario_type": "credit_card",
        "parameters": {
            "url": "https://e.creditcard.ecitic.com/citiccard/ebank-ocp/ebankpc/myaccount.html"
        }
    }


In [45]:
async with httpx.AsyncClient() as client:
    logger.info("发送请求到API网关")

    response = await client.post(
        "http://localhost:8000/tasks",
        json=request_data,
        timeout=300  # 5分钟超时，因为登录可能需要时间
    )

2025-02-28 19:49:04,542 - CreditCardTest - INFO - 发送请求到API网关
2025-02-28 19:49:09,853 - httpx - INFO - HTTP Request: POST http://localhost:8000/tasks "HTTP/1.1 200 OK"


In [50]:
request_data = {
            "url": "https://e.creditcard.ecitic.com/citiccard/ebank-ocp/ebankpc/myaccount.html"
}

In [52]:
async with httpx.AsyncClient() as client:
    logger.info("发送请求到API网关")

    response = await client.post(
        "http://localhost:8003/tools/browser/credit-card",
        json=request_data,
        timeout=300  # 5分钟超时，因为登录可能需要时间
    )

2025-02-28 20:12:28,535 - CreditCardTest - INFO - 发送请求到API网关
2025-02-28 20:13:11,500 - httpx - INFO - HTTP Request: POST http://localhost:8003/tools/browser/credit-card "HTTP/1.1 500 Internal Server Error"


In [53]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import socket
import time
import sys
import random
import threading
from datetime import datetime, timedelta


In [54]:
driver_path = '/usr/local/bin/chromedriver'
chrome_options = Options()
chrome_options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")

service = Service(driver_path)
driver = webdriver.Chrome(service=service, options=chrome_options)

In [19]:
login_indicators = [
    # 用户名和头像区域
    "div.user-info",
    ".user-avatar",
    "img.avatar",
    
    # 账户信息区域
    ".account-balance",
    ".card-balance",
    ".本期还定金额",
    
    # 具体信息区域
    "div:contains('本期还定金额')",
    "#myCredit",
    ".my-credit-card",
    
    # 用XPath直接定位包含用户名的元素
    "//*[contains(text(), '您好')]",
    "//*[contains(text(), '王')]",
    
    # 信用卡信息区域
    ".credit-card-info",
    ".card-number",
    
    # 卡号区域
    "div:contains('6226****9039')"
]

In [55]:
# 获取页面上所有可能的选择器路径
def get_element_selectors(driver):
    # 使用JavaScript来获取所有元素的选择器
    js_script = """
    function getPathTo(element) {
        if (element.id !== '')
            return '#' + element.id;
        if (element === document.body)
            return 'body';

        var ix = 0;
        var siblings = element.parentNode.childNodes;
        for (var i = 0; i < siblings.length; i++) {
            var sibling = siblings[i];
            if (sibling === element)
                return getPathTo(element.parentNode) + ' > ' + element.tagName.toLowerCase() + ':nth-child(' + (ix + 1) + ')';
            if (sibling.nodeType === 1 && sibling.tagName === element.tagName)
                ix++;
        }
    }
    
    var elements = document.querySelectorAll('*');
    var result = [];
    for (var i = 0; i < elements.length; i++) {
        var el = elements[i];
        if (el.textContent && el.textContent.trim() !== '' && el.style.display !== 'none') {
            result.push({
                text: el.textContent.trim().substring(0, 50),
                selector: getPathTo(el),
                classes: el.className,
                id: el.id
            });
        }
    }
    return result;
    """
    
    selectors = driver.execute_script(js_script)
    return selectors

selectors = get_element_selectors(driver)
print("页面上的元素选择器:")
for i, selector in enumerate(selectors):
    if i > 100:  # 限制输出数量
        print("...")
        break
    print(f"文本: {selector['text']}")
    print(f"选择器: {selector['selector']}")
    print(f"类名: {selector['classes']}")
    print(f"ID: {selector['id']}")
    print("---")

页面上的元素选择器:
文本: @charset "UTF-8";[ng\:cloak],[ng-cloak],[data-ng-c
选择器: #undefined > html:nth-child(1)
类名: 
ID: 
---
文本: @charset "UTF-8";[ng\:cloak],[ng-cloak],[data-ng-c
选择器: #undefined > html:nth-child(1) > head:nth-child(1)
类名: 
ID: 
---
文本: @charset "UTF-8";[ng\:cloak],[ng-cloak],[data-ng-c
选择器: #undefined > html:nth-child(1) > head:nth-child(1) > style:nth-child(1)
类名: 
ID: 
---
文本: WeChat/Weixin for Web
选择器: #undefined > html:nth-child(1) > head:nth-child(1) > title:nth-child(1)
类名: 
ID: 
---
文本: if(window.top!== window.self){top.location=self.lo
选择器: #undefined > html:nth-child(1) > head:nth-child(1) > script:nth-child(3)
类名: 
ID: 
---
文本: endian
                
            
            

选择器: body
类名: ng-scope ng-isolate-scope loaded
ID: 
---
文本: endian
                
            
            

选择器: body > div:nth-child(1)
类名: main
ID: 
---
文本: endian
                
            
            

选择器: body > div:nth-child(1) > div:nth-child(1)
类名: main_inner
ID: 
---
文本: endia

In [26]:
# 检测用户是否已登录
def is_user_logged_in(driver):
    # 最可靠的登录指示器
    primary_indicators = [
        "#userName",         # 用户名元素
        "#nameRare",         # 用户名称显示区域
        ".ca_num",           # 信用卡号区域
        "#cardList"          # 卡片列表区域
    ]
    
    # 尝试查找任一指示器
    for selector in primary_indicators:
        try:
            element = driver.find_element(By.CSS_SELECTOR, selector)
            if element.is_displayed():
                print(f"找到登录指示器: {selector}")
                return True
        except:
            pass
    
    # 如果上面的选择器都没找到，尝试通过文本内容检测
    login_text_indicators = [
        "本期仍需还款",
    ]
    
    for text in login_text_indicators:
        if text in driver.page_source:
            print(f"通过文本检测到登录状态: '{text}'")
            return True
    
    return False

In [27]:
is_user_logged_in(driver)

找到登录指示器: #userName


True

In [32]:
# 改进文本检测方法
def check_login_by_text(driver):
    # 使用JavaScript直接从DOM获取文本
    js_script = """
    const textContent = document.body.innerText;
    return {
        hasName: textContent.includes('欢迎您'),
        hasBill: textContent.includes('本期应还金额'),
        hasDate: textContent.includes('到期还款日')
    };
    """
    
    result = driver.execute_script(js_script)
    
    # 打印详细结果
    print("文本检测结果:")
    for key, value in result.items():
        print(f"  {key}: {value}")
    
    # 只要有一项为True，就认为已登录
    return any(result.values())

In [31]:
check_login_by_text(driver)

文本检测结果:
  hasBill: True
  hasDate: True
  hasName: True


True

In [35]:
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import re

def check_login_status(driver):
    """检查用户是否已登录"""
    # 方法1: 检查元素是否存在
    element_indicators = ["#userName", "#nameRare", ".ca_num", "#cardList"]
    for selector in element_indicators:
        try:
            element = driver.find_element(By.CSS_SELECTOR, selector)
            if element.is_displayed():
                print(f"元素检测: 找到登录指示器 {selector}")
                return True
        except:
            pass
    
    # 方法2: 检查文本内容
    js_script = """
    const textContent = document.body.innerText;
    return {
        hasName: textContent.includes('王') || textContent.includes('欢迎您'),
        hasBill: textContent.includes('本期应还金额'),
        hasDate: textContent.includes('到期还款日')
    };
    """
    
    result = driver.execute_script(js_script)
    if any(result.values()):
        print(f"文本检测: 找到登录指示 {result}")
        return True
    
    return False

def extract_account_info(driver):
    """提取账户信息"""
    # 使用JavaScript提取具体的文本值
    js_script = """
    const result = {
        welcomeMessage: null,
        billAmount: null,
        dueDate: null,
        cardNumber: null,
        minPayment: null
    };
    
    // 查找欢迎信息
    const welcomeElements = Array.from(document.querySelectorAll('*')).filter(el => 
        el.textContent && el.textContent.includes('欢迎您'));
    if (welcomeElements.length > 0) {
        result.welcomeMessage = welcomeElements[0].textContent.trim();
    }
    
    // 查找卡号
    const cardElements = document.querySelectorAll('.ca_num');
    if (cardElements.length > 0) {
        result.cardNumber = cardElements[0].textContent.trim();
    }
    
    // 查找账单金额
    const billElements = document.querySelectorAll('td:nth-child(1) > span.txt14');
    if (billElements.length > 0) {
        for (let el of billElements) {
            if (el.parentElement && el.parentElement.textContent.includes('本期应还金额')) {
                result.billAmount = el.textContent.trim();
                break;
            }
        }
    }
    
    // 查找最低还款金额
    const minPayElements = document.querySelectorAll('td:nth-child(2) > span.txt14');
    if (minPayElements.length > 0) {
        for (let el of minPayElements) {
            if (el.parentElement && el.parentElement.textContent.includes('最低还款金额')) {
                result.minPayment = el.textContent.trim();
                break;
            }
        }
    }
    
    // 查找到期还款日
    const dateElements = document.querySelectorAll('td:nth-child(2) > span.txt14');
    if (dateElements.length > 0) {
        for (let el of dateElements) {
            if (el.parentElement && el.parentElement.textContent.includes('到期还款日')) {
                result.dueDate = el.textContent.trim();
                break;
            }
        }
    }
    
    return result;
    """
    
    return driver.execute_script(js_script)

def extract_username_from_text(text):
    """从文本中提取用户名"""
    match = re.search(r'[欢迎您|您好]，([\w*]+)', text)
    return match.group(1) if match else "未知用户"

def display_formatted_info(data):
    """格式化显示账户信息"""
    # 提取用户名
    username = extract_username_from_text(data.get('welcomeMessage', ''))
    
    # 获取卡号
    card_number = data.get('cardNumber', '未获取到卡号')
    
    # 格式化显示
    print("\n" + "="*50)
    print("            中信银行信用卡账户信息摘要")
    print("="*50)
    
    print(f"\n👤 用户信息:")
    print(f"   用户名: {username}")
    print(f"   卡号: {card_number}")
    
    print(f"\n💰 账单信息:")
    print(f"   应还金额: ¥{data.get('billAmount', '未获取到')}")
    print(f"   最低还款: ¥{data.get('minPayment', '未获取到')}")
    print(f"   还款日期: {data.get('dueDate', '未获取到')}")
    
    print("\n" + "="*50)


In [36]:
def main(driver):
    """主函数: 检测登录状态并提取信息"""
    # 检查是否已登录
    is_logged_in = check_login_status(driver)
    
    if is_logged_in:
        print("用户已登录，正在提取账户信息...")
        # 提取账户信息
        account_info = extract_account_info(driver)
        # 格式化显示信息
        display_formatted_info(account_info)
        # 返回账户信息供进一步处理
        return account_info
    else:
        print("未检测到登录状态，请先登录账户。")
        return None

# 调用主函数


In [37]:
account_data = main(driver)

元素检测: 找到登录指示器 #userName
用户已登录，正在提取账户信息...

            中信银行信用卡账户信息摘要

👤 用户信息:
   用户名: 王*典
   卡号: 6226****9039

💰 账单信息:
   应还金额: ¥12707.37
   最低还款: ¥849.91
   还款日期: 2025-03-04



In [60]:
def search_and_send_message(driver, contact_name, message):
    """
    Search for a contact and send them a message in WeChat Web
    
    Args:
        driver: Selenium WebDriver instance
        contact_name: Name of the contact to search for and message
        message: Text message to send
    """
    try:
        # Click on the search input field (magnifying glass icon)
        search_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, ".search_bar input"))
        )
        search_button.click()
        
        # Input the contact name in the search field
        search_input = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".search_bar input"))
        )
        search_input.clear()
        search_input.send_keys(contact_name)
        time.sleep(2)  # Wait for search results
        
        # Find and click on the contact in search results
        search_results = driver.find_elements(By.CSS_SELECTOR, ".contact_item")
        contact_found = False
        
        for result in search_results:
            try:
                name_element = result.find_element(By.CSS_SELECTOR, ".nickname")
                if contact_name in name_element.text:
                    result.click()
                    contact_found = True
                    print(f"Found and clicked on contact: {contact_name}")
                    break
            except:
                continue
        
        if not contact_found:
            print(f"Contact {contact_name} not found in search results")
            return False
        
        # Wait for chat window to load
        time.sleep(2)
        
        # Find the message input area and send message
        edit_area = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "editArea"))
        )
        
        # Clear any existing text and input our message
        edit_area.clear()
        edit_area.send_keys(message)
        
        # Find and click the send button
        send_button = driver.find_element(By.CSS_SELECTOR, ".btn_send")
        send_button.click()
        
        print(f"Message sent to {contact_name}: {message}")
        return True
        
    except Exception as e:
        print(f"Error searching for contact and sending message: {str(e)}")
        return False

In [62]:

# Example usage
contact_to_message = "朱天阳"  # The contact you want to search for
message_text = "我来试试：这是大模型发出的内容"

search_and_send_message(driver, contact_to_message, message_text)

Found and clicked on contact: 朱天阳
Message sent to 朱天阳: 我来试试：这是大模型发出的内容


True